In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
df = pd.read_csv("creditcard.csv")
train, test = train_test_split(df, test_size=0.2, stratify=df["Class"], random_state=42)
normal = train[train["Class"] == 0].drop(columns="Class").to_numpy(dtype="float32")
x_train, x_val = train_test_split(normal, test_size=0.1, random_state=42)
x_test = test.drop(columns="Class").to_numpy(dtype="float32")
y_test = test["Class"].to_numpy()
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

In [ ]:
model = Sequential([Input((30,)),
    Dense(32, activation="relu"),
    Dense(8, activation="relu"),
    Dense(30)])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

In [ ]:
model.fit(x_train, x_train, epochs=30, batch_size=128, validation_data=(x_val, x_val))

In [ ]:
normal_error = np.mean((x_val - model.predict(x_val, verbose=0)) ** 2, axis=1)
threshold = np.percentile(normal_error, 99.5)

In [ ]:
test_error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=1)
anomalies = test_error > threshold
print("Test accuracy (%):", round(100 * accuracy_score(y_test, anomalies), 2))
print("Detection AUC:", roc_auc_score(y_test, test_error))